<a href="https://colab.research.google.com/github/akash-reddy-k/edgeAi/blob/phase%2Fsecond/edgeAi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ultralytics
!pip install kaggle
!pip install lap>=0.5.12

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 7.8 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = userdata.get('KAGGLE_TOKEN')
import kaggle
api = kaggle.api  # auto-authenticates on import using the token above

api.dataset_download_files(
    "nikanvasei/shanghaitech-campus-dataset-test",
    path="/content/drive/MyDrive/edgeai_project/data",
    unzip=True
)

Dataset URL: https://www.kaggle.com/datasets/nikanvasei/shanghaitech-campus-dataset-test


In [3]:
from google.colab import drive
drive.mount('/content/drive/edgeAi')

MessageError: Error: credential propagation was unsuccessful

In [4]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


In [5]:
from google.colab import files
uploaded = files.upload()  # opens a file picker — select your video
video_path = list(uploaded.keys())[0]

Saving TwoKids.mp4 to TwoKids.mp4


In [6]:
import time

# --- Config ---
owners_away = True  # simulate "house owners not home" — you'll wire this to a real toggle later
CONSECUTIVE_FRAMES_THRESHOLD = 5  # require detection across multiple frames to avoid false alerts
alert_counter = 0
alert_sent = False

def send_alert(person_count):
    # Placeholder — swap this for a real Telegram/webhook call
    print(f"🚨 ALERT: {person_count} person(s) detected while house is away!")

# --- Run inference frame-by-frame on your video ---
videoframes = model(video_path, stream=True, classes=[0], vid_stride=5, verbose=False)
for frame_result in videoframes:
    person_count = len(frame_result.boxes)

    if owners_away and person_count > 0:
        alert_counter += 1
        if alert_counter >= CONSECUTIVE_FRAMES_THRESHOLD and not alert_sent:
            send_alert(person_count)
            alert_sent = True
    else:
        alert_counter = 0
        alert_sent = False  # room is empty again — ready to alert on the next intrusion

🚨 ALERT: 2 person(s) detected while house is away!


In [7]:
import os
model_path = 'yolov8n.pt'
size_mb = os.path.getsize(model_path) / (1024 * 1024)
print(f"Model file size: {size_mb:.2f} MB")

model.export(format='onnx')  # creates yolov8n.onnx
onnx_size_mb = os.path.getsize('yolov8n.onnx') / (1024 * 1024)
print(f"ONNX model size: {onnx_size_mb:.2f} MB")

total_params = sum(p.numel() for p in model.model.parameters())
print(f"Total parameters: {total_params:,}")

Model file size: 6.25 MB
Ultralytics 8.4.142 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

PyTorch: starting from 'yolov8n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (6.2 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 12 packages in 310ms
Prepared 4 packages in 1.77s
Installed 4 packages in 257ms
 + colorama==0.4.6
 + onnx==1.22.0
 + onnxruntime==1.29.0
 + onnxslim==0.1.96

requirements: AutoUpdate success ✅ 2.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset 18...
ONNX: slimming with onnx

In [8]:
model.export(format='onnx', int8=True, data='coco8.yaml')  # data= needed for calibration during INT8 export

WARNING ⚠️ 'int8' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.142 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

PyTorch: starting from 'yolov8n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (6.2 MB)

ONNX: starting export with onnx 1.22.0 opset 18...
ONNX: slimming with onnxslim 0.1.96...
ONNX: collecting INT8 calibration images from 'data=coco8.yaml'

WARNING ⚠️ Dataset 'coco8.yaml' images not found, missing path '/content/datasets/coco8/images/val'
Unzipping /content/datasets/coco8.zip to /content/datasets/coco8...: 100% ━━━━━━━━━━━━ 25/25 2.9Kfiles/s 0.0s
Dataset download success ✅ (0.3s), saved to /content/datasets

val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 644.5±316.6 MB/s, size: 54.0 KB)
val: Scanning /content/datasets/coco8/labels/val... 4 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4/4

ONNX: export success ✅ 4.7s, saved as 'yolov8n_int8.onnx' (3.4 MB)

Export complete (5.1s)
Results saved to /content/yolov8n_int8.onnx
Predict:         yolo predict task=detect model=yolov8n_int8.onnx imgsz=640 
Validate:        yolo val task=detect model=yolov8n_int8.onnx imgsz=640 data=coco.yaml  
Visualize:       https://netron.app


'yolov8n_int8.onnx'